# Lab 2: The Value Object & The Forward Pass

## Introduction

Every computation in micrograd creates a "story" that records:
- **What number was produced** (`self.data`)
- **Which nodes were involved** (`self._prev`) ← the "family tree"
- **What operation was used** (`self._op`)

This story is called the **COMPUTATION GRAPH**.

When you write `c = a + b`, Python calls `a.__add__(b)`, which:
1. Computes `c.data = a.data + b.data`
2. Remembers `c._prev = {a, b}` ← "c was born from a and b"
3. Remembers `c._op = '+'` ← "using addition"

The computation graph is what makes automatic backprop possible (Lab 4). For now we only implement the forward direction — gradients come later.

## Exercise: Implement the Value Class (Forward Pass Only)

### Your Task

Complete the `Value` class below by implementing `__init__`, `__add__`, and `__mul__`:

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        # TODO: self.data  = float(data)
        # TODO: self.grad  = 0.0             ← gradient starts at zero
        # TODO: self._prev = set(_children)  ← parents in the computation graph
        # TODO: self._op   = _op             ← operation that created this node
        # TODO: self.label = label            ← optional human-readable name
        # TODO: self._backward = lambda: None ← placeholder (used in Lab 4)
        raise NotImplementedError("Implement __init__")

    def __add__(self, other):
        # Wrap plain numbers so  val + 2  works as well as  val + Value(2)
        other = other if isinstance(other, Value) else Value(other)
        # TODO: return Value(self.data + other.data, (self, other), '+')
        raise NotImplementedError("Implement __add__")

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # TODO: return Value(self.data * other.data, (self, other), '*')
        raise NotImplementedError("Implement __mul__")

    # ── Already implemented — these build on __add__ and __mul__ ─────────────
    def __radd__(self, other):     return self + other       # other + self
    def __rmul__(self, other):     return self * other       # other * self
    def __neg__(self):             return self * -1          # -self
    def __sub__(self, other):      return self + (-other)    # self - other
    def __rsub__(self, other):     return other + (-self)    # other - self
    def __truediv__(self, other):  return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def __repr__(self):
        lbl = f"'{self.label}' " if self.label else ''
        return f"Value({lbl}data={self.data:.4f}, grad={self.grad:.4f})"

### Correct Answer

<details>
<summary>Click to reveal solution</summary>

```python
def __init__(self, data, _children=(), _op='', label=''):
    self.data  = float(data)
    self.grad  = 0.0
    self._prev = set(_children)
    self._op   = _op
    self.label = label
    self._backward = lambda: None

def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    return Value(self.data + other.data, (self, other), '+')

def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    return Value(self.data * other.data, (self, other), '*')
```
</details>

## Demo: Building a Computation Graph

Let's build the expression: `d = (a * b) + c`

In [ ]:
# Build the expression: d = (a * b) + c
a = Value(2.0,  label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')

e = a * b;  e.label = 'e'   # e = 2 × (-3) = -6
d = e + c;  d.label = 'd'   # d = (-6) + 10 = 4

print(f"a = {a}")
print(f"b = {b}")
print(f"e = a * b = {e}")
print(f"d = e + c = {d}")

### Test Your Implementation

Verify the computation graph structure:

In [ ]:
# ── Verify the family tree ────────────────────────────────────────────────
assert d.data == 4.0,  f"d.data should be 4.0, got {d.data}"
assert e in d._prev,   "e should be in d._prev"
assert c in d._prev,   "c should be in d._prev"
assert a in e._prev,   "a should be in e._prev"
assert b in e._prev,   "b should be in e._prev"
print("✓ Family tree checks: PASS")

print(f"\nd._op = '{d._op}'   (expected '+')")
print(f"e._op = '{e._op}'   (expected '*')")

print("\n✓ Forward pass: PASS")

## Visualize the Computation Graph

Let's visualize the computation graph we just built using graphviz.

The graph will show:
- **Nodes**: Each Value with its data and gradient
- **Operations**: The operations (+, *, etc.) connecting nodes
- **Flow**: How data flows from inputs (a, b, c) to output (d)

In [ ]:
from utils import visualize_graph

# Visualize the computation graph for d = (a * b) + c
visualize_graph(d, title="Computation Graph: d = (a * b) + c")

**What you should see:**
- Nodes labeled 'a', 'b', 'c', 'e', 'd' with their data values
- Operation nodes '*' and '+' connecting them
- Arrows showing the flow: a and b → * → e, then e and c → + → d
- Gradients are all 0.0 (we haven't done backprop yet!)

**Reading the graph:**
- Left side: Input values (a, b, c)
- Middle: Intermediate computation (e = a * b)
- Right side: Final output (d = e + c)

## Edge Cases to Test

Test more complex expressions:

In [ ]:
# Edge case 1: Chained operations
x = Value(2.0)
y = Value(3.0)
z = Value(4.0)
result = (x + y) * z
assert result.data == 20.0, f"Expected 20.0, got {result.data}"
print("✓ Chained operations: PASS")

# Edge case 2: Using plain numbers
a = Value(5.0)
b = a + 3  # Should work even though 3 is not a Value
assert b.data == 8.0, f"Expected 8.0, got {b.data}"
print("✓ Mixed Value and number: PASS")

# Edge case 3: Multiple operations on same value
x = Value(2.0)
y = x + x  # x used twice
assert y.data == 4.0, f"Expected 4.0, got {y.data}"
assert len(y._prev) == 2, "y should have 2 parents (both are x)"
print("✓ Reusing values: PASS")

# Edge case 4: Negative numbers
a = Value(-5.0)
b = Value(3.0)
c = a * b
assert c.data == -15.0, f"Expected -15.0, got {c.data}"
print("✓ Negative numbers: PASS")

print("\n🎉 All edge cases passed!")

## Bonus: Implement `__pow__`

Try implementing the power operator so that `val ** 2` works:

In [ ]:
# Add this method to your Value class above:
# def __pow__(self, other):
#     assert isinstance(other, (int, float))
#     return Value(self.data ** other, (self,), f'**{other}')

# Then test:
# x = Value(3.0)
# y = x ** 2
# assert y.data == 9.0
# print("✓ Power operator: PASS")

## Summary

You've learned:
- How to build a computation graph that tracks operations
- The `Value` class stores data, gradients, and parent relationships
- Every operation creates a new node with `_prev` and `_op`

Next: Lab 3 will manually compute gradients using the chain rule!